# 📦 Demand Forecasting & Inventory Optimization Engine


## 🧠 Business Problem
Predict Units Sold per (Store, Product, Date)                                                                                                               
→ Optimize inventory replenishment                                                                                                                          
→ Reduce stockouts and overstock

## Milestone 1: Data Collection, Preprocessing & EDA

In [2]:
#  IMPORT LIBRARIES

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings('ignore')



### 1. Setup & Data Loading

In [3]:
# LOAD DATA
df = pd.read_csv('../data/raw/retail_store_inventory.csv')

In [3]:
# Preview
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,1/1/2022,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,1/1/2022,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,1/1/2022,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,1/1/2022,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,1/1/2022,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


### 2. Data Understanding

In [5]:
# Structure
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  object 
 1   Store ID            73100 non-null  object 
 2   Product ID          73100 non-null  object 
 3   Category            73100 non-null  object 
 4   Region              73100 non-null  object 
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  object 
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  object 
dtypes: float64(3), int64(5), object(7)
memory usage: 8.4+

In [6]:

# Summary statistics
df.describe()


,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Holiday/Promotion,Competitor Pricing
count,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000
mean,274.469877,136.464870,110.004473,141.494720,55.135108,10.009508,0.497305,55.146077
std,129.949514,108.919406,52.277448,109.254076,26.021945,7.083746,0.499996,26.191408
min,50.000000,0.000000,20.000000,-9.990000,10.000000,0.000000,0.000000,5.030000
25%,162.000000,49.000000,65.000000,53.670000,32.650000,5.000000,0.000000,32.680000
50%,273.000000,107.000000,110.000000,113.015000,55.050000,10.000000,0.000000,55.010000
75%,387.000000,203.000000,155.000000,208.052500,77.860000,15.000000,1.000000,77.820000
max,500.000000,499.000000,200.000000,518.550000,100.000000,20.000000,1.000000,104.940000


In [7]:
# Check columns
df.columns

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast',
       'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion',
       'Competitor Pricing', 'Seasonality'],
      dtype='object')

Key Observations
-Target Variable: Sales (or equivalent demand column)
Granularity: Likely daily per product per store
Typical Features:
Date
Store ID
Product ID
Sales (target)
Price, Promotion, Inventory (if available)
Business Problem (Data Terms)

Predict:

Sales_t = f(Product, Store, Time, Price, Promotions, Lagged Demand)

→ Goal: Minimize stockouts & overstock via accurate demand forecasts.

### 3. Data Cleaning

In [4]:
# Rename columns for consistency and ease of use
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [5]:
df.isnull().sum()

date                  0
store_id              0
product_id            0
category              0
region                0
inventory_level       0
units_sold            0
units_ordered         0
demand_forecast       0
price                 0
discount              0
weather_condition     0
holiday/promotion     0
competitor_pricing    0
seasonality           0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [7]:
# FIX DATA TYPES
df['date'] = pd.to_datetime(df['date'])

In [9]:
print(df['date'].dtype)

datetime64[ns]


In [11]:
# Sort for time series integrity
df = df.sort_values(['store_id', 'product_id', 'date'])


### 3. FEATURE ENGINEERING

In [10]:
# Time Features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['weekday'] = df['date'].dt.weekday
df['weekofyear'] = df['date'].dt.isocalendar().week
df['is_weekend'] = df['weekday'].isin([5,6]).astype(int)

In [13]:
# Lag Features (Critical)
group_cols = ['store_id', 'product_id']

for lag in [1, 7, 14, 30,60]:
    df[f'lag_{lag}'] = df.groupby(group_cols)['units_sold'].shift(lag)

In [12]:
# Rolling Features
# Average demand over last 7 days (excluding current day)

df['rolling_mean_7'] = df.groupby(group_cols)['units_sold'].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

df['rolling_mean_14'] = df.groupby(group_cols)['units_sold'].transform(
    lambda x: x.shift(1).rolling(14).mean()
)

df['rolling_std_7'] = df.groupby(group_cols)['units_sold'].transform(
    lambda x: x.shift(1).rolling(7).std()
)

In [14]:
# Price Dynamics
# Compare my price vs competitor
df['price_diff'] = df['price'] - df['competitor_pricing']
df['discount_effect'] = df['price'] * df['discount']

# Optional (helps stability)
df['price_ratio'] = df['price'] / (df['competitor_pricing'] + 1e-5)

print(df['price_diff'].describe())
print(df['discount_effect'].describe())



count    73100.000000
mean        -0.010969
std          2.888538
min         -5.000000
25%         -2.530000
50%          0.000000
75%          2.500000
max          5.000000
Name: price_diff, dtype: float64
count    73100.000000
mean       552.153511
std        505.044373
min          0.000000
25%        142.000000
50%        420.800000
75%        870.600000
max       2000.000000
Name: discount_effect, dtype: float64


In [15]:
# REMOVE LEAKAGE
if 'demand_forecast' in df.columns:
    df.drop(columns=['demand_forecast'], inplace=True)

In [16]:
df.isnull().sum()

date                     0
store_id                 0
product_id               0
category                 0
region                   0
inventory_level          0
units_sold               0
units_ordered            0
price                    0
discount                 0
weather_condition        0
holiday/promotion        0
competitor_pricing       0
seasonality              0
year                     0
month                    0
weekday                  0
weekofyear               0
is_weekend               0
rolling_mean_7         700
rolling_mean_14       1400
rolling_std_7          700
lag_1                  100
lag_7                  700
lag_14                1400
lag_30                3000
lag_60                6000
price_diff               0
discount_effect          0
price_ratio              0
dtype: int64

In [17]:
# Drop NA after lagging
df.dropna(inplace=True)

In [18]:
df = df.reset_index(drop=True)

In [19]:
print("Feature Engineering Completed")
print("Shape:", df.shape)
print(df.head())

Feature Engineering Completed
Shape: (67100, 30)
        date store_id product_id     category region  inventory_level  \
0 2022-03-02     S001      P0001    Groceries  South              216   
1 2022-03-03     S001      P0001         Toys  South              304   
2 2022-03-04     S001      P0001         Toys  South              338   
3 2022-03-05     S001      P0001    Furniture   East              496   
4 2022-03-06     S001      P0001  Electronics  South              463   

   units_sold  units_ordered  price  discount  ... rolling_mean_14  \
0          93            120  58.52        15  ...      158.428571   
1          36             44  14.01         0  ...      138.428571   
2         284             43  33.06        15  ...      138.785714   
3         282            167  27.67        10  ...      157.142857   
4         192             54  62.37        10  ...      175.357143   

   rolling_std_7  lag_1  lag_7  lag_14  lag_30  lag_60  price_diff  \
0     105.205694   28

In [21]:
# save cleaned data
import os

os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/retail_store_inventory_cleaned.csv', index=False)